# Weitere Objektorientierte Konzepte

## Statische Elemente

* Statische Elemente sind der Klasse zugeordnet und **keine** Attribute einer Instanz dieser Klasse
  * Genauer: Statische Elemente sind Elemente des Klassenobjektes
* Der Zugriff erfolgt über die Klasse mit dem üblichen Punkt-Operator
* Zur Definition eines statischen Attributs wird innerhalb der Klasse einer Variablen ein Wert zugewiesen

In [1]:
class Person:
    number_of_created_people = 0

    def __init__(self, name):
        self.name = name
        Person.number_of_created_people += 1

p1 = Person('A')
p2 = Person('B')
Person.number_of_created_people


2

* statische Methoden sind etwas komplizierter, dafür wird ein soganannter Decorator genutzt
  * Decorators werden im nächsten Abschnitt genauer besprochen
  * In Java wird dieses Konzept mit `Annotations` umgesetzt

In [3]:
class Person:
    number_of_created_people = 0
    def __init__(self, name):
        self.name = name
        Person.number_of_created_people += 1
    
    @classmethod
    def info(cls):
        return f'class {cls.__name__} has {cls.number_of_created_people} instances'


p1 = Person('A')
p2 = Person('B')
Person.info()


'class Person has 2 instances'

## Abstrakte Klassen

* Eine abstrakte Klasse definiert Attribute und Methoden, kann aber nicht instanziert werden
    * der Konstruktor kann nicht aufgerufen werden
* Damit können in einem Modell Klassen eingeführt werden, die an der Spitze einer Klassenhierarchie stehen, aber in der Realität keine Entsprechung haben
* Abstrakte Klassen enthalten Methoden, die in einer Subklasse implementiert werden **müssen**
    * dafür wird der Decorator `@abstractmethod` aus dem Modul `abc` genutzt

* Beispiel
    * Der Begriff "Musikinstrument" ist jedem geläufig, aber es gibt keine Objekte, die ein "pures" Musikinstrument repräsentieren
    * Es gibt nur konkrtete Instrumente: Gitarre, Violine, ...
        * diese sind aber definitiv allgemein Instrumente

* Falsche Implementierung
    * Wir erzeugen ein Instrument, das gar keine Töne von sich geben kann

In [8]:
class Instrument:
    def __init__(self, manufacturer):
        self.manufacturer = manufacturer


class Guitar(Instrument):
    def __init__(self, manufacturer, number_of_strings):
        super().__init__(manufacturer)
        self.number_of_strings = number_of_strings
    def play(self):
        print('strum')

class Violin(Instrument):
    def __init__(self, manufacturer):
        super().__init__(manufacturer)
    def play(self):
        print('fiddel')

instrument = Instrument('noone') # ????
guitar = Guitar('Fender', 6)
violin = Violin('Stradivari')
# instrument.play() # impossible, what is the sound of an instrument???
guitar.play()
violin.play()

strum
fiddel


In [11]:
from abc import ABC, abstractmethod
class Instrument(ABC):
    def __init__(self, manufacturer):
        self.manufacturer = manufacturer
    @abstractmethod
    def play(self):
        pass


class Guitar(Instrument):
    def __init__(self, manufacturer, number_of_strings):
        super().__init__(manufacturer)
        self.number_of_strings = number_of_strings
    def play(self):
        print('strum')

class Violin(Instrument):
    def __init__(self, manufacturer):
        super().__init__(manufacturer)
    def play(self):
        print('fiddel')

# instrument = Instrument('noone') # now it is impossible to use Instrument constructor
guitar = Guitar('Fender', 6)
violin = Violin('Stradivari')
guitar.play()
violin.play()


strum
fiddel


* Hinweis
    * Einfach eine leere `play`-Methode vorzusehen ist nicht korrekt, da wir dann wieder Instrument erzeugen könnten, die "nichts" machen, das ist sinnlos

## Mehrfachvererbung und Mixins

### Mehrfachvererbung

* Es gibt Situationen, in denen eine Klasse sinnvoll **mehrere** Basisklassen haben kann
* Beispiel
  * House, Boat, Houseboat

In [9]:
class House:
    def __init__(self, number_of_rooms):
        self.number_of_rooms = number_of_rooms

class Boat:
    def __init__(self, max_speed):
        self.max_speed = max_speed

class Houseboat(House, Boat):
    pass

* Vorsicht: So funktioniert das nicht, Mehrfachvererbung wird dadurch komplex, dass wir die Initialisierung beider Superklassen befriedigen müssen!

In [10]:
house = House(4)
boat = Boat(7.2)
houseboat = Houseboat()

TypeError: House.__init__() missing 1 required positional argument: 'number_of_rooms'

* Korrekte Implementierung

In [ ]:
class Houseboat(House, Boat):
    def __init__(self, number_of_rooms, max_speed):
        super().__init__(number_of_rooms) # works because House is the first Superclass
        Boat.__init__(self, max_speed) # constructor is accessible using __init__ as static method

houseboat = Houseboat(4, 7.2) 

### Mixins

* Bei Mehrfachvererbung haben die Basisklasen eine konkrete Bedeutung
    * es gibt Häuser, Boote und Hausboote
* Gruppiert man jedoch Attribute und Methoden zusammen, die **keine sinnvolle** isolierte Bedeutung haben, spricht man von einem Mixin
    * Ein Mixin wird man deshalb nie instanzieren

In [25]:
class PositionMixin:
    def __init__(self):
        self.position = (0,0,0)
    def set_position(self, x=0, y=0, z=0):
        self.position = (x,y,z)

class MoveableMixin:
    def move(self, x=0, y=0, z=0):
        self.position = tuple(a+b for a,b in zip(self.position, (x,y,z)))

class Car(PositionMixin, MoveableMixin):
    def __init__(self, max_speed):
        self.max_speed = max_speed
        PositionMixin.__init__(self)
        MoveableMixin.__init__(self)

car = Car(130)
print(car.position)
car.set_position(10,10,0)
print(car.position)
car.move(1,2,3)
print(car.position)


(0, 0, 0)
(10, 10, 0)
(11, 12, 3)


* Der Unterschied zwischen Mixin und Klasse ist eine reine Konvention in der Verwendung
    * Eine Namenskonvention kann dies verdeutlichen, `Moveable`**`Mixin`**